## 1) Install

In [1]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip -q install transformers open_clip_torch pandas numpy pillow scikit-learn tqdm requests

## 2) Imports & Config

In [5]:
import os, io, math, random, re, requests
from typing import Any, Dict

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import DistilBertTokenizerFast, DistilBertModel
import open_clip
from sklearn.metrics import average_precision_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cpu"
print("Device:", DEVICE)

CSV_PATH = "base_features_with_image.csv"

TEXT_MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256

# ✅ Only 4 numeric features now
NUMERIC_FEATURES = [
    "price",
    "msrp",
    "price_anomaly_z",
    "clip_image_text_sim"
]

THRESHOLD = 0.65

Device: cpu


## 3) Load Data & Build Text

In [9]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.decomposition import PCA
from PIL import Image
import requests, io
from tqdm import tqdm

# ===== SETTINGS =====
CSV_PATH = "base_features_with_image.csv"
DEVICE = "cpu"
HEADERS = {"User-Agent": "Mozilla/5.0"}  # prevents Amazon blocking

# ===== LOAD DATA =====
df = pd.read_csv(CSV_PATH)
df["title"] = df["title"].astype(str).fillna("")
df["description"] = df["description"].astype(str).fillna("")
df["text_concat"] = (df["title"] + " " + df["description"]).str.strip()

# ===== LOAD FAST CLIP MODEL =====
model = SentenceTransformer("sentence-transformers/clip-ViT-B-16", device=DEVICE)

# Encode text (normalize so cosine works properly)
text_emb = model.encode(
    df["text_concat"].tolist(),
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Encode images
image_emb_list = []
for url in tqdm(df["image_url"], desc="Processing Images"):
    try:
        r = requests.get(url, headers=HEADERS, timeout=8)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content)).convert("RGB")
        emb = model.encode(img, convert_to_tensor=True, normalize_embeddings=True)
    except:
        emb = torch.zeros(text_emb.shape[1])  # fallback vector
    image_emb_list.append(emb)

image_emb = torch.stack(image_emb_list)

# ===== DIMENSION REDUCE 512 → 64 USING PCA =====
pca = PCA(n_components=64)

text_emb_64 = torch.tensor(pca.fit_transform(text_emb.cpu()))
image_emb_64 = torch.tensor(pca.transform(image_emb.cpu()))

# normalize again after PCA to keep cosine meaningful
text_emb_64 = text_emb_64 / text_emb_64.norm(dim=-1, keepdim=True)
image_emb_64 = image_emb_64 / image_emb_64.norm(dim=-1, keepdim=True)

# Cosine similarity (diagonal per product)
similarities = (text_emb_64 * image_emb_64).sum(dim=1).numpy()

df["clip_image_text_sim"] = similarities

# === Add label_suspicious according to the following rule: ===
# Rule 1: Very low price compared to MSRP  
#         → price_anomaly_z < -1.5

# Rule 2: Image–Text mismatch  
#         → clip_image_text_sim < 0.25  

# Rule 3: Extremely low price  
#         → price < 0.35 * msrp

def compute_label(row):
    rule1 = row["price_anomaly_z"] < -1.5
    rule2 = row["clip_image_text_sim"] < 0.25
    rule3 = (row["price"] < 0.35 * row["msrp"]) if row["msrp"] > 0 else False

    return 1 if (rule1 or rule2 or rule3) else 0

df["label_suspicious"] = df.apply(compute_label, axis=1)
df.to_csv(CSV_PATH, index=False)
print("✅ CSV updated successfully!")
print("🎯 Embeddings are now 64-dim and similarity values are non-zero.")

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Batches:   0%|          | 0/459 [00:00<?, ?it/s]

Processing Images: 100%|███████████████████████████████████████████████████████| 29339/29339 [4:42:11<00:00,  1.73it/s]


✅ CSV updated successfully!
🎯 Embeddings are now 64-dim and similarity values are non-zero.


## 4) Clip image text sim column generated

In [6]:
df = pd.read_csv(CSV_PATH)
df.head()

,title,description,category,price,msrp,price_anomaly_z,image_url,clip_image_text_sim,label_suspicious
0,Lee posh Lactic Acid 60% Anti ageing Pigmenta...,PROFESSIONAL GRADE Face Peel: this peel stimul...,Skin Care,799.0,2000.0,-0.242392,https://images-na.ssl-images-amazon.com/images...,-0.304340,1
1,Branded SLB Works New 1.5mm Titanium 1200 nee...,Item name: 1.5mm titanium 1200 needles microne...,Skin Care,2040.0,2040.0,0.266590,https://images-na.ssl-images-amazon.com/images...,0.256523,0
2,Generic 1 Pc brand snail eye cream remove dar...,"Use: eye, item type: cream, net wt: 20g, gzzz:...",Skin Care,1042.0,1824.0,-0.142728,https://images-na.ssl-images-amazon.com/images...,-0.179685,1
3,Generic Anti Snoring Snore Stopper Sleep Apne...,Prevent the tongue from dropping backward or b...,Skin Care,1399.0,2185.0,0.003691,https://images-na.ssl-images-amazon.com/images...,-0.027819,1
4,Harveys Crunchy & Creame Gourmet Delicacies C...,Harvey's wafer Cream Wafer 110g. Made in India,Grocery & Gourmet Foods,570.0,594.0,-0.040384,https://images-na.ssl-images-amazon.com/images...,0.179403,1
